# MusicFM FP32 candidate from original weights
Mac, kernel SongFormer Python3.11, root Section-Annotations.
Tidak melatih model, mengubah amplitudo audio, mengganti NaN, atau menimpa SongMap.
Mengekspor hanya encoder layer10 dari bobot asli, dengan input conv_in/rotary
seperti model aplikasi. Frontend dan rotary diperiksa terpisah. Konversi FP32
lebih besar dan bisa lebih lambat; ini kandidat baseline numerik sebelum kompresi.


In [ ]:
import sys, subprocess, json
from pathlib import Path
from datetime import datetime
assert Path('finetuning/config.json').is_file()
SONGMAP_ROOT = Path('/Users/danendra/Documents/ADA26/C6/SongMap')
records = sorted(Path('outputs.local').glob('musicfm-diagnostic-*/diagnostic.json'),
    key=lambda p:p.stat().st_mtime, reverse=True)
assert records, 'Jalankan musicfm_failure_diagnostic.ipynb dahulu'
diagnostic = records[0].parent
output = Path('outputs.local') / ('musicfm-fp32-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
print('Source diagnostic:',diagnostic)
print('Output:',output)

def run_logged(command, log_path):
    with log_path.open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in process.stdout:
                print(line,end='',flush=True)
                log.write(line)
                log.flush()
            code=process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if code:
        raise RuntimeError(f'Proses gagal ({code}); lihat {log_path}. Laporan parsial mungkin ada di {output}.')


Bandingkan frontend/native encoder, ekspor encoder FP32, lalu uji pada window Pilihanku. Jika pemeriksaan gagal, berhenti dan bagikan repair_report.json; jangan jalankan validasi berikutnya.

In [ ]:
run_logged([sys.executable,'-u','-m','deployment.export_musicfm_fp32',
    '--diagnostic-dir',str(diagnostic),'--songmap-root',str(SONGMAP_ROOT),
    '--output-dir',str(output)],Path(str(output)+'.export.log'))
repair = json.loads((output/'repair_report.json').read_text())
for key in ('wrapper_vs_native_encoder','frontend_conv_app_vs_original',
            'rotary_effect_same_conv','old_coreml_vs_torch_same_inputs',
            'fp32_coreml_vs_torch_same_inputs','fp32_source_rotary_vs_torch',
            'fp32_native_frontend_vs_torch'):
    print(key, json.dumps(repair[key]))
print('Exact window passed:',repair['exact_window_passed'])


Evaluasi encoder FP32 sebagai override terpisah. Frontend dan rotary aplikasi tetap, sehingga perubahan encoder compute terisolasi. Fitur kandidat memakai cache baru; cache lama tidak diubah.

In [ ]:
proofs=sorted(Path('outputs.local').glob('coreml-comparison-*/fp32-validation.json'),
    key=lambda p:p.stat().st_mtime,reverse=True)
proofs=[p for p in proofs if (lambda r:r.get('full_validation') and r.get('logit_parity_passed')
    and r.get('precision_metadata')=='fp32')(json.loads(p.read_text()))]
assert proofs
proof=proofs[0]
cache=output/'validation-cache'

def evaluate_repaired(limit,name):
    run_logged([sys.executable,'-u','-m','deployment.evaluate_pipeline',
        '--songmap-root',str(SONGMAP_ROOT),'--fp32-validation-report',str(proof),
        '--musicfm-export-report',str(output/'repair_report.json'),
        '--cache-dir',str(cache),'--limit',str(limit),'--output',str(output/f'{name}.json')],
        output/f'{name}.log')

evaluate_repaired(1,'one-track')
print(json.dumps(json.loads((output/'one-track.json').read_text()),indent=2))


Setelah satu lagu lolos runtime, jalankan semua validation. FP32 encoder bisa lebih lambat daripada kandidat lama. Cache kandidat yang sudah selesai dipakai ulang.

In [ ]:
evaluate_repaired(0,'all-validation')
r=json.loads((output/'all-validation.json').read_text())
print('Reference F1:',r['reference_cache_with_fp32_head']['macro_f1_present'])
print('Candidate F1:',r['coreml_encoders_with_fp32_head']['macro_f1_present'])
print('Delta:',r['macro_f1_delta'],'| Section agreement:',r['section_prediction_agreement'])
print('Changed sections:',len(r['changed_sections']))
print('Report:',output/'all-validation.json')
